# Lasso — L1-Regularized Regression (Turkey, Pipeline B)

**Model**: `sklearn.linear_model.Lasso` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: YES (StandardScaler, fit on train fold only) / **HP tuning**: YES — `LassoCV(cv=TimeSeriesSplit(5))` on train+val, freeze alpha  
**n_lags**: 4 / **COVID**: passed through unscaled via `split_for_scaler()`

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, flatten_data, mean_fill_dataset,
    get_features, split_for_scaler, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42
np.random.seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
VAL_END     = '2017-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
N_LAGS      = 4
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0}

print('Lasso (Turkey) — Cat3 variables, n_lags={}, scaling=YES, HP tune=YES'.format(N_LAGS))

Lasso (Turkey) — Cat3 variables, n_lags=4, scaling=YES, HP tune=YES


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)
print('Features: {} total ({} scaled, {} COVID passed through)'.format(
    len(features), len(scaled_cols), len(unscaled_cols)))

Features: 25 total (22 scaled, 3 COVID passed through)


In [3]:
# Phase A: HP tuning on train+val (1995-2017) via TimeSeriesSplit
print('Phase A: HP tuning on train+val (1995-2017)...')

tune_data   = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat   = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat   = tune_flat.loc[tune_flat.date.dt.month.isin([3, 6, 9, 12]), :]
tune_flat   = tune_flat.dropna(axis=0, how='any').reset_index(drop=True)

feature_cols = [
    c for c in tune_flat.columns
    if c != 'date' and c != TARGET
    and any(c == f or c.startswith(f + '_') for f in features)
]
X_tune = tune_flat[feature_cols].values
y_tune = tune_flat[TARGET].values

# Scale tune set — fit scaler once on train+val (no test leakage)
scaler_A     = StandardScaler()
X_tune_scaled = scaler_A.fit_transform(X_tune)

alpha_path = np.logspace(-6, 0, 100)
tscv       = TimeSeriesSplit(n_splits=5)
lasso_cv   = LassoCV(alphas=alpha_path, cv=tscv, random_state=SEED,
                     max_iter=20000, n_jobs=1)  # n_jobs=1: Windows multiprocessing constraint
lasso_cv.fit(X_tune_scaled, y_tune)
best_alpha = lasso_cv.alpha_
print('Best alpha: {:.4g}'.format(best_alpha))

Phase A: HP tuning on train+val (1995-2017)...


Best alpha: 0.00163


In [4]:
# Phase B: Rolling test loop with frozen alpha
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions  = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q   = pd.Timestamp(q_date)
    actual = monthly[monthly['date'] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)
        train_mask = (monthly['date'] >= TRAIN_START) & (monthly['date'] <= forecast_date)
        train = monthly[train_mask].reset_index(drop=True)

        train_masked = gen_lagged_data(metadata, train.copy(), forecast_date, lag=0)
        train_filled = mean_fill_dataset(train_masked, train_masked)
        train_flat   = flatten_data(train_filled, TARGET, N_LAGS)
        train_flat   = train_flat.loc[train_flat.date.dt.month.isin([3, 6, 9, 12]), :]
        train_flat   = train_flat.dropna(axis=0, how='any').reset_index(drop=True)

        if len(train_flat) < 10:
            predictions[vintage_name].append(np.nan)
            continue

        # Separate scaled / unscaled (COVID) feature columns
        sc_cols = [c for c in feature_cols if any(c == s or c.startswith(s + '_') for s in scaled_cols)]
        us_cols = [c for c in feature_cols if any(c == u or c.startswith(u + '_') for u in unscaled_cols)]

        try:
            scaler      = StandardScaler()
            X_train_sc  = scaler.fit_transform(train_flat[sc_cols].values)
            X_train     = np.hstack([X_train_sc, train_flat[us_cols].values]) if us_cols else X_train_sc
            y_train     = train_flat[TARGET].values

            model = Lasso(alpha=best_alpha, max_iter=20000, random_state=SEED)
            model.fit(X_train, y_train)

            # Build test row
            test_row    = monthly[monthly['date'] == forecast_date].reset_index(drop=True)
            test_masked = gen_lagged_data(metadata, test_row.copy(), forecast_date, lag=0)
            test_filled = mean_fill_dataset(train_masked, test_masked)
            ctx    = train_filled.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb    = pd.concat([ctx, test_filled], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            test_flat = cmb_fl.tail(1)

            X_test_sc = scaler.transform(test_flat[sc_cols].values)
            X_test    = np.hstack([X_test_sc, test_flat[us_cols].values]) if us_cols else X_test_sc
            pred = model.predict(X_test)[0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vintage_name].append(pred)

    if (i + 1) % 8 == 0:
        print('  {} / {} quarters'.format(i + 1, len(test_quarters)))

print('Done. {} quarters, {} failures.'.format(len(test_quarters), failed))

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

  32 / 32 quarters
Done. 32 quarters, 0 failures.


C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

In [5]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        'date': test_quarters, 'actual': actuals_list,
        'prediction': predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, 'lasso_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))

Saved lasso_m1.csv (32 rows) pred=[-0.0627,0.0362]
Saved lasso_m2.csv (32 rows) pred=[-0.0522,0.0468]
Saved lasso_m3.csv (32 rows) pred=[-0.0124,0.0256]


In [6]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print('Lasso (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))

Lasso (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.011512  MAE=0.009926  N=8
  m2  RMSFE=0.013821  MAE=0.012703  N=8
  m3  RMSFE=0.014414  MAE=0.011680  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.097328  MAE=0.064901  N=7
  m2  RMSFE=0.085565  MAE=0.052474  N=7
  m3  RMSFE=0.063314  MAE=0.039348  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.019434  MAE=0.014994  N=16
  m2  RMSFE=0.022992  MAE=0.017571  N=16
  m3  RMSFE=0.013467  MAE=0.009714  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.048244  MAE=0.025197  N=32
  m2  RMSFE=0.043889  MAE=0.024069  N=32
  m3  RMSFE=0.032011  MAE=0.016788  N=32
